Predict new genes

In [ ]:
import torch
from openpyxl.workbook import Workbook
# SVMs predict new regeneration genes
import json
import pickle
import random
from collections import Counter
import sys
sys.path.append('/home/wwd/codebox/ODT-main')
import joblib
import numpy as np
import openpyxl
from sklearn import svm
from sklearn.model_selection import GridSearchCV, LeaveOneOut
from torch import nn
from d2l import torch as d2l
import torch
from torch.nn import functional as F
from tqdm import tqdm
# 已知的正负样本
negative = ['SMESG000050058','SMESG000022746','SMESG000010270','SMESG000017894','SMESG000042630','SMESG000025574',
            'SMESG000037117','SMESG000006969','SMESG000032691','SMESG000066574']
positive = ['SMESG000005389', 'SMESG000046293', 'SMESG000063577', 'SMESG000020104', 'SMESG000060238', 'SMESG000029273', 'SMESG000053725',
            'SMESG000070594', 'SMESG000070103', 'SMESG000049472', 'SMESG000016858', 'SMESG000060978', 'SMESG000037081', 'SMESG000016751',
            'SMESG000056159', 'SMESG000045786', 'SMESG000070150', 'SMESG000064333', 'SMESG000025543', 'SMESG000060148', 'SMESG000020739',
            'SMESG000042345', 'SMESG000057772', 'SMESG000033647', 'SMESG000036986', 'SMESG000060356', 'SMESG000030852', 'SMESG000044282',
            'SMESG000025390', 'SMESG000061064', 'SMESG000074748', 'SMESG000042131', 'SMESG000074761']
new_p=['SMESG000019132','SMESG000010938','SMESG000081615','SMESG000075225'] # 第一轮测试出的再生基因
new_N=['SMESG000081365','SMESG000075453','SMESG000074160','SMESG000071310','SMESG000070953'] # 第一轮测试出的非再生基因
positive+=new_p
negative+=new_N
print(len(positive),positive)
print(len(negative),negative)


In [ ]:

def get_data():
    # with open('./cluster_model/gene_embeddings.pkl', 'rb') as f:
    with open('../result/gene_embeddings.pkl', 'rb') as f:
        gene_ebedding = pickle.load(f)
    positive_data=[] # 确定为正样本
    negative_data=[] # 确定为负样本
    unknown_data=[]  # 不确定
    for gene in negative:
        if(gene in gene_ebedding):
            negative_data.append({'gene_id':gene,'label':'negative','tensor':[float(item) for item in gene_ebedding[gene]]})
    for gene in positive:
        if (gene in gene_ebedding):
            positive_data.append({'gene_id': gene, 'label': 'positive', 'tensor': [float(item) for item in gene_ebedding[gene]]})
    for gene in gene_ebedding:
        if(gene not in positive and gene not in negative):
            unknown_data.append({'gene_id': gene, 'label': 'unknown', 'tensor': [float(item) for item in gene_ebedding[gene]]})
    # print(len(positive_data)) 34
    # print(len(negative_data)) 13
    # print(len(unknown_data)) 15117
    return positive_data, negative_data, unknown_data

In [ ]:
def getMinSS_SMESG(ss,data_dict):
    negative_items=[]
    for data in data_dict:
        if(data_dict[data]['ss']<=ss):
            data=data.split('-')
            data=data[0]
            negative_items.append(data)
    return negative_items
def getSS_SMESG(data_dict,neg_lenth=10,pos_lenth=10):
    negative_items=[]
    positive_items=[]
    data_dict_list=[]
    for data in data_dict:
        data_dict_list.append({'ss':data_dict[data]['ss'],'gene_id':data})
    data_list=sorted(data_dict_list,key=lambda x:x['ss'])
    for data in data_list[:neg_lenth]:
        data=data['gene_id']
        data = data.split('-')
        data=data[0]
        negative_items.append(data)
    for data in data_list[-1-pos_lenth:-1]:
        data = data['gene_id']
        data = data.split('-')
        data = data[0]
        positive_items.append(data)
    return negative_items,positive_items
def SVC_analyse(ss_max,id):
    ebeddings=[]
    labels=[]
    positive_data, negative_data, unknown_data = get_data()
    for data in positive_data:
        ebeddings.append(data['tensor'])
        labels.append(1)
    for data in negative_data:
        ebeddings.append(data['tensor'])
        labels.append(0)
    #with open('./cluster_model/gene_embedding.pkl', 'rb') as f:
    with open('../result/gene_embeddings.pkl', 'rb') as f:
        gene_ebedding = pickle.load(f)
    with open('../dataSets/gene_data_allIndents_ss.pkl', 'rb') as f:
        data_dict = pickle.load(f)
    # print(len(data_dict))
    negative_items=getMinSS_SMESG(ss_max,data_dict)
    # negative_items,positive_items=getSS_SMESG(data_dict,id*5+5,int((id*3+5)/3))
    negative_items=[item for item in negative_items if item not in negative]
    negative_items_len=len(negative_items)
    print('new_negative',len(negative_items))
    for data in negative_items:
        if(data in gene_ebedding):
            ebeddings.append(gene_ebedding[data])
        labels.append(0)
    # for data in positive_items:
    #     if(data in gene_ebedding):
    #         ebeddings.append([float(item) for item in gene_ebedding[data]])
    #     labels.append(1)
    ebeddings=torch.tensor(ebeddings)
    labels=torch.tensor(labels)
    # 创建SVM分类器对象
    clf = svm.SVC()
    # parameters = {'kernel': ('linear', 'rbf','poly'), 'C': [1, 10]}
    parameters = {'kernel': ('linear', 'rbf','poly'), 'C': [0.001,0.005, 0.01,0.05, 0.1,0.5, 1,5, 10,50, 100,500, 1000]}
    clf = GridSearchCV(clf, parameters)
    loo = LeaveOneOut()
    # 训练模型
    for train_index, test_index in loo.split(ebeddings):
        ebeddings_train, ebeddings_test = ebeddings[train_index], ebeddings[test_index]
        labels_train,labels_test = labels[train_index],labels[test_index]
        clf.fit(ebeddings_train, labels_train)
        # 测试模型
        true_num=0
        predictions = clf.predict(ebeddings_test)
        for i in range(len(predictions)):
            if predictions[i] == labels_test[i]:
                true_num += 1
    print('true_num',true_num,'all',len(predictions))
    predictions = clf.predict(ebeddings)
    true_num=0
    for i in range(len(predictions)):
        if predictions[i]==labels[i]:
            true_num+=1
    print('true_num',true_num,'all',len(predictions),'accuracy',float(true_num)/len(predictions))
    joblib.dump(clf, '../models/SVM_models/'+str(id)+'_0_svm_model.pkl')
    all_ebeddings=[]
    gene_ids=[]
    gene_result={}
    for data in unknown_data:
        all_ebeddings.append(data['tensor'])
        gene_ids.append(data['gene_id'])
    results=clf.predict(all_ebeddings)
    for i in tqdm(range(len(results))):
        result=results[i]
        if(result==1):
            gene_result[gene_ids[i]]='positive'
        else:
            gene_result[gene_ids[i]] = 'negative'
    print(Counter(results))
    with open('../models/SVM_models/'+str(id)+'_result_gene_classification2.pkl', 'wb')as f:
        pickle.dump(gene_result,f)
    return Counter(results),true_num,len(predictions), negative_items_len
#
# SVC_analyse(0.018)
def SVM_20(start,lenth,target_path):
    SVM_result={}
    for i in range(21):
        print('id',i,i*lenth+start)
        results,true_num,predictions,add_negtive=SVC_analyse(i*lenth+start,i)
        results=dict(results)
        if('1' in results):
            SVM_result[i]={'positive':int(results[1]),'negative':int(results[0]),'add_negative_nums':add_negtive,'accuracy':float(true_num)/predictions,'ss':i*lenth+start,'id':str(i)}
            print('positive',int(results[1]))
        else:
            SVM_result[i] = {'positive': 0, 'negative': int(results[0]),'add_negative_nums':add_negtive,
                             'accuracy': float(true_num) / predictions,'ss':i*lenth+start, 'id': str(i)}
    with open(target_path, "w") as file:
        json.dump(SVM_result, file)

In [ ]:
# SVM_20(start=0.022,lenth=0.0001,target_path="../models/SVM_models/result_new.json") # 计算新增的positive,negative
# # 提取新再生基因
new_grow_gene={}
new_grow_gene_scores=[]
for id in range(21):
    with open('../models/SVM_models/'+str(id)+'_result_gene_classification2.pkl', 'rb')as f:
        gene_result=pickle.load(f)
        new_grow_gene[id]=[]
        for gene in gene_result:
            if(gene_result[gene]=='positive'):
                new_grow_gene[id].append(gene)
                new_grow_gene_scores.append(gene)
for id in new_grow_gene:
    print('id:',id,'genes:',new_grow_gene[id])
print(Counter(new_grow_gene_scores))
with open('../models/SVM_models/SVM_gene_classification_new.json', 'w')as f:
    json.dump(new_grow_gene,f)
new_grow_gene_score=dict(Counter(new_grow_gene_scores))
print(new_grow_gene_score)

disturb

In [ ]:

import os
import pickle

GPU_NUMBER = [6, 2, 3, 4, 5, 0]
os.environ["CUDA_VISIBLE_DEVICES"] = ",".join([str(s) for s in GPU_NUMBER])
from collections import Counter
import datasets
from geneformer import InSilicoPerturber
from geneformer import InSilicoPerturberStats
from geneformer import EmbExtractor

modelpath = '../models/time_finetune'
dataset_path = '../models/disturb_models/data.datasets'
datasets_P = datasets.Dataset.load_from_disk(dataset_path)
print(datasets_P)

# 实例化 InSilicoPerturber 对象，用于模拟基因删除实验，以确定在扩张型心肌病 (dcm) 状态下，
# 哪些基因的删除会显著使嵌入向非衰竭 (nf) 状态转移
embex = EmbExtractor(model_type="CellClassifier",
                     num_classes=7,
                     filter_data={"cell_type": ["WT", "0h", "12h", "1.5d", "3d", "10d", "5d"]},
                     max_ncells=40000,
                     emb_layer=-1,
                     summary_stat="exact_mean",
                     forward_batch_size=16,
                     nproc=16)
state_embs_dict = embex.get_state_embs(cell_states_to_model={
    'state_key': 'cell_type',
    'start_state': 'WT',
    'goal_state': '10d',
    'alt_states': ["0h", "12h", "1.5d", "3d", "5d"]}
    , model_directory=modelpath
    , input_data_file=dataset_path
    , output_directory="disturb_models"  # 输出目录路径
    , output_prefix="output_prefix")
# with open('/home/wwd/codebox/Geneformer/new/planaria2/disturb_models/output_prefix.pkl','rb')as f1:
#     data=pickle.load(f1)
# print(data)
# 实例化 InSilicoPerturber 对象
print("state_embs_dict", state_embs_dict)
isp = InSilicoPerturber(
    perturb_type="delete",  # 扰动类型
    perturb_rank_shift=None,  # 扰动等级变化
    genes_to_perturb="all",  # 要扰动的基因
    combos=0,  # 组合数量
    anchor_gene=None,  # 锚定基因
    model_type="CellClassifier",  # 模型类型
    num_classes=7,  # 类别数量
    emb_mode="cell",  # 嵌入模式
    cell_emb_style="mean_pool",  # 细胞嵌入风格
    filter_data={"cell_type": ["WT", "0h", "12h", "1.5d", "3d", "10d", "5d"]},  # 过滤数据
    cell_states_to_model={
        'state_key': 'cell_type',
        'start_state': 'WT',
        'goal_state': '10d',
        'alt_states': ["0h", "12h", "1.5d", "3d", "5d"]},
    state_embs_dict=state_embs_dict,
    max_ncells=20000,  # 最大细胞数量
    emb_layer=-1,  # 嵌入层
    forward_batch_size=2,  # 前向批处理大小
    nproc=1  # 处理器数量
)

# 输出体外扰动的中间文件
isp.perturb_data(
    modelpath,  # 模型路径
    dataset_path,  # 输入数据路径
    "disturb_models",  # 输出目录路径
    "output_prefix"  # 输出前缀
)

# 实例化 InSilicoPerturberStats 对象
ispstats = InSilicoPerturberStats(
    mode="goal_state_shift",  # 模式
    genes_perturbed="all",  # 被扰动的基因
    combos=0,  # 组合数量
    anchor_gene=None,  # 锚定基因
    cell_states_to_model={
        'state_key': 'cell_type',
        'start_state': 'WT',
        'goal_state': '10d',
        'alt_states': ["0h", "12h", "1.5d", "3d", "5d"]}
)

# 从中间文件中提取数据并处理统计信息，输出最终的 .csv 文件
ispstats.get_stats(
    "disturb_models",  # 输入数据路径
    None,  # 替代输入数据路径
    "disturb_models",  # 输出目录路径
    "output_prefix"  # 输出前缀
)


Predict relatives between genes

In [ ]:

import pandas as pd
import itertools
from src.models import Graph_predicate


def makexlsx(target_path,data_dict):
    # print(data_dict)
    # 创建一个新的工作簿
    wb = Workbook()
    # 选择活动的工作表
    ws = wb.active
    # 给工作表命名
    ws.title = "gene_all"
    # 添加数据到工作表中
    ws.append(  ['Regenerative_genes', 'source_positive_gene', 'score','source_negative_gene','score','target_positive_gene','score', 'target_negative_gene','score' ])
    for R_gene in data_dict:
        for i in range(200):
            ws.append([R_gene,data_dict[R_gene]['source_positive'][i][0],data_dict[R_gene]['source_positive'][i][1],data_dict[R_gene]['source_negative'][i][0],data_dict[R_gene]['source_negative'][i][1]
                       ,data_dict[R_gene]['target_positive'][i][0],data_dict[R_gene]['target_positive'][i][1],data_dict[R_gene]['target_negative'][i][0],data_dict[R_gene]['target_negative'][i][1]])

    # 保存工作簿为 Excel 文件
    wb.save(target_path)
    pass

def edge_class(edge_weight):
    if (edge_weight<-100):
        return 0
    if (edge_weight>=-100 and edge_weight<-15):
        return 1
    if (edge_weight>=-15 and edge_weight<-3):
        return 2
    if (edge_weight>=-3 and edge_weight<=3):
        return 3
    if (edge_weight>3 and edge_weight<15):
        return 4
    if (edge_weight>=15 and edge_weight<100):
        return 5
    if (edge_weight>=100):
        return 6
def edge_target():
    # 此方法用于生成图
    edge_pure = []
    # edge_classes=[]
    with open('../result/edges.csv') as f:
        edges = f.readlines()
        for edge in edges[1:]:
            edge = edge.strip('\n')
            edge = edge.split(',')
            edge_pure.append([edge[1], edge[2], edge_class(float(edge[3]))])
    return edge_pure
    pass
def analyseRegulate_gene():
    edge_predicate = edge_target()
    # device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    device = torch.device("cuda:9")
    Graph_predicate_example = Graph_predicate(gene_tensor_path='../result/gene_embeddings.pkl',
                                              gene_disturb_path='../result/planaria_disturb.pkl',
                                              target_edges=edge_predicate, device=device)
    # Graph_predicate_example.load_AEmodel('./model_only_gene/ae_model_final.pth')
    # 经过实验，没有经过分结果分类处理的效果很差，使用AE生成向量的结果很难收敛，使用两种向量直接拼接的效果更好，但有限
    # tensor_model=Graph_predicate_example.get_Model_tensor()
    tensor_model = Graph_predicate_example.get_Tensor()
    model_dict = {'input_size': 548, 'num_classes': 7, 'hidden_sizes': [640, 1280, 640], 'num_epochs': 5000,
                  'dropout': 0.3}
    tensors = list(tensor_model.values())
    model_dict['input_size'] = tensors[0].shape[0] * 2
    print(model_dict['input_size'])
    # Graph_predicate_example.MLPmodel(model_dict)
    Graph_predicate_example.loadMLPmodel('../models/MLP_model/mlp_model_checkpoint_epoch_1000.pth', model_dict)
    # 加载新发现的再生基因
    new_regenerative_genes = ['SMESG000019132', 'SMESG000081615', 'SMESG000075225', 'SMESG000010938', 'SMESG000077295',
                              'SMESG000021611']
    gene_paths = {}
    for new_regenerative_gene in new_regenerative_genes:
        print(new_regenerative_gene)
        print(model_dict)
        gene_paths[new_regenerative_gene] = {'source_positive': [], 'target_positive': [], 'target_negative': [],
                                             'source_negative': []}
        # 先考虑再生基因在头的情况
        source_tensors = []
        gene_names = []
        for gene in Graph_predicate_example.gene_names:
            if (gene != new_regenerative_gene):
                gene_names.append(gene)
                source_tensors.append(torch.cat((Graph_predicate_example.gene_model_tensors[new_regenerative_gene],
                                                 Graph_predicate_example.gene_model_tensors[gene]), dim=0))
        source_tensors = torch.stack(source_tensors).to(device)
        mean = source_tensors.mean(dim=1, keepdim=True)  # 形状为 (n, 1)
        std = source_tensors.std(dim=1, keepdim=True)  # 形状为 (n, 1)

        # 标准化输入
        epsilon = 1e-8  # 避免除以零
        inputs_standardized = (source_tensors - mean) / (std + epsilon)  # 结果形状为 (n, m)

        target_tensors = Graph_predicate_example.model_prdicate('MLP_model', inputs_standardized)
        outs = torch.argmax(target_tensors, dim=1)
        print(outs)
        targets = outs.tolist()
        # labels = [0, 1, 2, 3, 4, 5, 6]
        # for tensor in target_tensors.tolist():
        #     # print(tensor)
        #     scores = 0
        #     for i in range(len(tensor)):
        #         scores += tensor[i] * labels[i]
        #     targets.append(scores)
        gene_target = dict(zip(gene_names, targets))
        gene_target = sorted(gene_target.items(), key=lambda item: item[1])
        # print(gene_target)
        gene_paths[new_regenerative_gene]['source_positive'] = [gene_data for gene_data in gene_target[-300:]]
        gene_paths[new_regenerative_gene]['source_negative'] = [gene_data for gene_data in gene_target[:300]]
        # 再考虑再生基因在尾的情况
        source_tensors = []
        gene_names = []
        for gene in Graph_predicate_example.gene_names:
            if (gene != new_regenerative_gene):
                gene_names.append(gene)
                source_tensors.append(torch.cat((Graph_predicate_example.gene_model_tensors[gene],
                                                 Graph_predicate_example.gene_model_tensors[new_regenerative_gene]),
                                                dim=0))
        source_tensors = torch.stack(source_tensors).to(device)
        mean = source_tensors.mean(dim=1, keepdim=True)  # 形状为 (n, 1)
        std = source_tensors.std(dim=1, keepdim=True)  # 形状为 (n, 1)

        # 标准化输入
        epsilon = 1e-8  # 避免除以零
        inputs_standardized = (source_tensors - mean) / (std + epsilon)  # 结果形状为 (n, m)

        target_tensors = Graph_predicate_example.model_prdicate('MLP_model', inputs_standardized)
        outs = torch.argmax(target_tensors, dim=1)
        print(outs)
        targets = outs.tolist()
        gene_target = dict(zip(gene_names, targets))
        gene_target = sorted(gene_target.items(), key=lambda item: item[1])
        # print(gene_target)
        gene_paths[new_regenerative_gene]['target_positive'] = [gene_data for gene_data in gene_target[-300:]]
        gene_paths[new_regenerative_gene]['target_negative'] = [gene_data for gene_data in gene_target[:300]]
    # makexlsx('./link_result/gene_link5.xlsx', gene_paths)
    print('saved')
    test_genes = ['SMESG000014324', 'SMESG000037557', 'SMESG000017064']
    test_results = {}

    for new_regenerative_gene in new_regenerative_genes:
        source_tensors = []
        for test_gene in test_genes:
            source_tensors.append(torch.cat((Graph_predicate_example.gene_model_tensors[new_regenerative_gene],
                                             Graph_predicate_example.gene_model_tensors[test_gene]), dim=0))
        source_tensors = torch.stack(source_tensors).to(device)
        mean = source_tensors.mean(dim=1, keepdim=True)  # 形状为 (n, 1)
        std = source_tensors.std(dim=1, keepdim=True)  # 形状为 (n, 1)
        # 标准化输入
        epsilon = 1e-8  # 避免除以零
        inputs_standardized = (source_tensors - mean) / (std + epsilon)  # 结果形状为 (n, m)
        target_tensors = Graph_predicate_example.model_prdicate('MLP_model', inputs_standardized)
        targets = []
        labels = [0, 1, 2, 3, 4, 5, 6]
        for tensor in target_tensors.tolist():
            # print(tensor)
            scores = 0
            for i in range(len(tensor)):
                scores += tensor[i] * labels[i]
            targets.append(scores)
        gene_target = dict(zip(test_genes, targets))
        test_results[new_regenerative_gene] = {}
        test_results[new_regenerative_gene]['source'] = gene_target
        source_tensors = []
        for test_gene in test_genes:
            source_tensors.append(torch.cat((Graph_predicate_example.gene_model_tensors[test_gene],
                                             Graph_predicate_example.gene_model_tensors[new_regenerative_gene]), dim=0))
        source_tensors = torch.stack(source_tensors).to(device)
        mean = source_tensors.mean(dim=1, keepdim=True)  # 形状为 (n, 1)
        std = source_tensors.std(dim=1, keepdim=True)  # 形状为 (n, 1)
        # 标准化输入
        epsilon = 1e-8  # 避免除以零
        inputs_standardized = (source_tensors - mean) / (std + epsilon)  # 结果形状为 (n, m)
        target_tensors = Graph_predicate_example.model_prdicate('MLP_model', inputs_standardized)
        targets = []
        labels = [0, 1, 2, 3, 4, 5, 6]
        for tensor in target_tensors.tolist():
            # print(tensor)
            scores = 0
            for i in range(len(tensor)):
                scores += tensor[i] * labels[i]
            targets.append(scores)
        gene_target = dict(zip(test_genes, targets))
        test_results[new_regenerative_gene]['target'] = gene_target
        print(new_regenerative_gene, test_results[new_regenerative_gene])
def getRegulateAll():
    # 创建新的 Excel 工作簿和工作表
    # 使用模型预测所有基因的调控关系
    edge_predicate = edge_target()
    # device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    device = torch.device("cuda:9")
    # 模型加载
    Graph_predicate_example = Graph_predicate(gene_tensor_path='../result/gene_embeddings.pkl',
                                              gene_disturb_path='../result/planaria_disturb.pkl',
                                              target_edges=edge_predicate, device=device)
    # Graph_predicate_example.load_AEmodel('./model_only_gene/ae_model_final.pth')
    # 经过实验，没有经过分结果分类处理的效果很差，使用AE生成向量的结果很难收敛，使用两种向量直接拼接的效果更好，但有限
    # tensor_model=Graph_predicate_example.get_Model_tensor()
    tensor_model = Graph_predicate_example.get_Tensor()
    model_dict = {'input_size': 548, 'num_classes': 7, 'hidden_sizes': [640, 1280, 640], 'num_epochs': 5000,
                  'dropout': 0.3}
    tensors = list(tensor_model.values())
    model_dict['input_size'] = tensors[0].shape[0] * 2
    print(model_dict['input_size'])
    # Graph_predicate_example.MLPmodel(model_dict)
    Graph_predicate_example.loadMLPmodel('../models/MLP_model/mlp_model_checkpoint_epoch_1000.pth', model_dict)
    # 加载新发现的再生基因
    all_genes = Graph_predicate_example.gene_model_tensors.keys()
    # print(all_genes)
    # print(len(all_genes))
    # 初始化
    batch_size = 12  # 一次处理 12 组基因对
    save_interval = 100000  # 每 10 万次保存一次
    batch_results = []  # 缓存批量结果
    save_count = 0  # 记录已保存次数

    # 设置模型为评估模式
    Graph_predicate_example.mlpModel.eval()

    # 组合基因对并批量计算
    with torch.no_grad():
        gene_pairs = list(itertools.combinations(all_genes, 2))  # 预生成所有组合
        total_pairs = len(gene_pairs)  # 总基因对数量

        for i in tqdm(range(0, total_pairs, batch_size), total=total_pairs // batch_size):
            batch_pairs = gene_pairs[i: i + batch_size]  # 取 batch_size 组
            source_tensors = []

            for gene1, gene2 in batch_pairs:
                tensor = torch.cat((
                    Graph_predicate_example.gene_model_tensors[gene1],
                    Graph_predicate_example.gene_model_tensors[gene2]
                ), dim=0).unsqueeze(0)  # 增加 batch 维度
                source_tensors.append(tensor)

            # 转换为批量张量
            source_tensors = torch.cat(source_tensors, dim=0).to(device)

            # 标准化处理
            mean = source_tensors.mean(dim=1, keepdim=True)
            std = source_tensors.std(dim=1, keepdim=True)
            inputs_standardized = (source_tensors - mean) / (std + 1e-8)

            # 预测输出
            target_tensors = Graph_predicate_example.model_prdicate('MLP_model', inputs_standardized)
            outs = torch.argmax(target_tensors, dim=1).tolist()  # 转换为列表

            # 过滤符合条件的基因对
            for (gene1, gene2), out in zip(batch_pairs, outs):
                if abs(out - 3) >= 2:
                    batch_results.append([gene1, gene2, out])

            # 定期保存
            if len(batch_results) >= save_interval:
                df = pd.DataFrame(batch_results, columns=['Gene1', 'Gene2', 'Score'])
                df.to_csv(f'../result/link_result/gene_link_part_{save_count}.csv', index=False)
                save_count += 1
                batch_results = []  # 清空缓存
                print(save_count,'saved')

    # 保存剩余数据
    if batch_results:
        df = pd.DataFrame(batch_results, columns=['Gene1', 'Gene2', 'Score'])
        df.to_csv(f'../result/link_result/gene_link_part_{save_count}.csv', index=False)
    pass

In [ ]:
# GrnmAe_model.load_state_dict()
edge_predicate = edge_target()
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = torch.device("cuda:9")
Graph_predicate_example = Graph_predicate(gene_tensor_path='../result/gene_embeddings.pkl',gene_disturb_path='../result/planaria_disturb.pkl',target_edges=edge_predicate,device=device)
# Graph_predicate_example.load_AEmodel('./model_only_gene/ae_model_final.pth')
# 经过实验，没有经过分结果分类处理的效果很差，使用AE生成向量的结果很难收敛，使用两种向量直接拼接的效果更好
tensor_model=Graph_predicate_example.get_Tensor()
model_dict={'input_size':548,'num_classes':7,'hidden_sizes':[640,1280,640],'num_epochs':1000,'dropout':0.3}
tensors=list(tensor_model.values())
model_dict['input_size'] =tensors[0].shape[0]*2
print(model_dict['input_size'] )
Graph_predicate_example.loadMLPmodel('../models/MLP_model2/mlp_model_checkpoint_epoch_5000.pth', model_dict)
Graph_predicate_example.MLPmodel(model_dict)
tensor_model=Graph_predicate_example.get_Tensor()
model_dict={'input_size':548,'num_classes':7,'hidden_sizes':[1024,512,512],'num_epochs':50000,'dropout':0.1}
tensors=list(tensor_model.values())
model_dict['input_size'] =tensors[0].shape[0]*2
print(model_dict['input_size'])
Graph_predicate_example.MLPmodel(model_dict)
def getGenes():
    gene_lists = []
    for i in tqdm(range(275)):
        csv_path = f'../result/link_result/gene_link_part_{i}.csv'
        with open(csv_path, 'r') as f:
            data_lists = f.readlines()
        for data_list in data_lists:
            data_list = data_list.split(',')
            gene_lists.append(data_list[0])
            gene_lists.append(data_list[1])
        gene_lists = list(set(gene_lists))
    df = pd.DataFrame(gene_lists, columns=['Gene'])  

    df.to_csv(f'../result/link_result/gene_list.csv', index=False)
    print(len(gene_lists))

def makeGeneEdge():
    # 加载新发现的再生基因
    new_regenerative_genes=['SMESG000019132','SMESG000081615','SMESG000075225','SMESG000010938','SMESG000077295','SMESG000021611']
    gene_id_positive = ['SMESG000005389', 'SMESG000046293', 'SMESG000063577', 'SMESG000020104', 'SMESG000060238', 'SMESG000029273', 'SMESG000053725', 'SMESG000070594', 'SMESG000070103', 'SMESG000049472', 'SMESG000016858', 'SMESG000060978', 'SMESG000037081', 'SMESG000016751', 'SMESG000056159', 'SMESG000045786', 'SMESG000070150', 'SMESG000064333', 'SMESG000025543', 'SMESG000060148', 'SMESG000020739', 'SMESG000042345', 'SMESG000057772', 'SMESG000033647', 'SMESG000036986', 'SMESG000060356', 'SMESG000030852', 'SMESG000044282', 'SMESG000025390', 'SMESG000061064', 'SMESG000074748', 'SMESG000042131', 'SMESG000049972', 'SMESG000074761', 'SMESG000072824']
    # 已知的调控因子
    regulatory_factor_genes=[]
    with open('../dataSets/edges.csv','r')as file:
        data_lists = file.readlines()
        for data_list in data_lists[1:]:
            data_list = data_list.strip('\n')
            data_list = data_list.split(',')
            regulatory_factor_genes.append(data_list[1])
            regulatory_factor_genes.append(data_list[2])
    regulatory_factor_genes = list(set(regulatory_factor_genes))
    print(len(regulatory_factor_genes))
    # 候选再生基因
    # 读取xlsx文件
    df = pd.read_excel('../result/GenesWithScores.xlsx')
    print(df.head())
    candidate_genes = df['gene_name'].tolist()
    print(candidate_genes)
    # print(regulatory_factor_genes)
    all_genes =[]
    all_genes += new_regenerative_genes
    all_genes += gene_id_positive
    all_genes += regulatory_factor_genes
    all_genes += candidate_genes
    all_genes = list(set(all_genes))
    print('all_genes',len(all_genes))
    gene_edges = []
    for gene1 in all_genes:
        all_genes2=[]
        source_tensors = []
        for gene2 in all_genes:
            if(gene1 == gene2):
                continue
            try:
                source_tensors.append(torch.cat((Graph_predicate_example.gene_model_tensors[gene1],Graph_predicate_example.gene_model_tensors[gene2]),dim=0))
                all_genes2.append(gene2)
            except:
                # print('gene1',gene1)
                # print('gene2',gene2)
                pass
        if(len(source_tensors)==0):
            continue
        source_tensor =  torch.stack(source_tensors).to(device)
        mean =source_tensor.mean(dim=1, keepdim=True)  # 形状为 (n, 1)
        std = source_tensor.std(dim=1, keepdim=True)  # 形状为 (n, 1)
        # 标准化输入
        epsilon = 1e-8  # 避免除以零
        inputs_standardized = (source_tensor - mean) / (std + epsilon)  # 结果形状为 (n, m)
        inputs_standardized =inputs_standardized .float()
        target_tensors =  Graph_predicate_example.model_prdicate('MLP_model', inputs_standardized)
        targets = []
        labels = [0, 1, 2, 3, 4, 5, 6]
        for tensor in target_tensors:
            # print(tensor)
            # scores = 0
            # for i in range(len(tensor)):
            #     scores += tensor[i] * labels[i]
            # targets.append(scores)
            targets.append(torch.argmax(tensor).item())
        for gene3 in all_genes2:
            if(targets[all_genes2.index(gene3)]>2.5 and targets[all_genes2.index(gene3)]<3.5):
                # 调控效果不明显，略过
                continue
            gene_edges.append([gene1,gene3,targets[all_genes2.index(gene3)]-3])
    with open('../result/gene_edges.txt','w')as file:
        for item in gene_edges:
            file.write(item[0])
            file.write(' ')
            file.write(item[1])
            file.write(' ')
            file.write(str(item[2]))
            file.write('\n')
    print(len(gene_edges))
    pass
def makeGeneNodes():
    # 加载新发现的再生基因
    new_regenerative_genes=['SMESG000019132','SMESG000081615','SMESG000075225','SMESG000010938','SMESG000077295','SMESG000021611']
    gene_id_positive = ['SMESG000005389', 'SMESG000046293', 'SMESG000063577', 'SMESG000020104', 'SMESG000060238', 'SMESG000029273', 'SMESG000053725', 'SMESG000070594', 'SMESG000070103', 'SMESG000049472', 'SMESG000016858', 'SMESG000060978', 'SMESG000037081', 'SMESG000016751', 'SMESG000056159', 'SMESG000045786', 'SMESG000070150', 'SMESG000064333', 'SMESG000025543', 'SMESG000060148', 'SMESG000020739', 'SMESG000042345', 'SMESG000057772', 'SMESG000033647', 'SMESG000036986', 'SMESG000060356', 'SMESG000030852', 'SMESG000044282', 'SMESG000025390', 'SMESG000061064', 'SMESG000074748', 'SMESG000042131', 'SMESG000049972', 'SMESG000074761', 'SMESG000072824']
    # 已知的调控因子
    regulatory_factor_genes=[]
    with open('../result/edges.csv','r')as file:
        data_lists = file.readlines()
        for data_list in data_lists[1:]:
            data_list = data_list.strip('\n')
            data_list = data_list.split(',')
            regulatory_factor_genes.append(data_list[1])
            regulatory_factor_genes.append(data_list[2])
    regulatory_factor_genes = list(set(regulatory_factor_genes))
    print(len(regulatory_factor_genes))
    # 候选再生基因
    # 读取xlsx文件
    df = pd.read_excel('../result/GenesWithScores.xlsx')

    candidate_genes = df['gene_name'].tolist()


    # 依此制作重要基因的结点文件，以pickle格式保存
    gene_all=[]
    with open('../result/gene_edges.txt', 'r') as file:
        gene_lists = file.readlines()
        for gene_list in gene_lists:
            gene_list = gene_list.strip('\n')
            gene_list = gene_list.split(' ')
            gene_all.append(gene_list[0])
    gene_all = list(set(gene_all))
    # 基因结点信息包括，基因在大模型的嵌入信息，基因的扰动信息, 基因的其他版本序号，基因的同源物
    # 读取嵌入向量的字典
    gene_tensor_path='../result/gene_embeddings.pkl'
    gene_disturb_path='../result/planaria_disturb.pkl'
    with open(gene_tensor_path, 'rb') as f1:
        gene_tensor = pickle.load(f1)
    with open(gene_disturb_path, 'rb') as f2:
        gene_disturb = pickle.load(f2)

    with open('../result/gene_features.csv','r')as f3:
        gene_features = f3.readlines()
        gene_features_str = gene_features[0]
        gene_features_str = gene_features_str.strip('\n')
        gene_features_str = gene_features_str.replace('"','')
        gene_features_str = gene_features_str.split(',')
        gene_features_data = {}
        for gene_feature in gene_features[1:]:
            gene_feature = gene_feature.strip('\n')
            gene_feature = gene_feature.replace('"','')
            gene_feature = gene_feature.split(',')
            gene_name = gene_feature[0]
            gene_name = gene_name.split('-')
            gene_name = gene_name[0]
            if(gene_name not in gene_features_data):
                gene_features_data[gene_name] = gene_feature
            else:
                new_feature =[]
                for i in range(len(gene_feature)):
                    if(i<=len(gene_features_data[gene_name])-1 and gene_feature[i] in gene_features_data[gene_name][i]):
                        new_feature.append(gene_features_data[gene_name][i])
                    elif(i<=len(gene_features_data[gene_name])-1 ):
                        new_feature.append(gene_features_data[gene_name][i]+'_'+gene_feature[i])
                    else:
                        new_feature.append(gene_feature[i])
                for j in range(i,len(gene_features_data[gene_name])):
                    new_feature.append(gene_features_data[gene_name][j])
                gene_features_data[gene_name] = new_feature

    print(len(gene_features_data))
    # print(gene_features_data.keys())
    gene_nodes = {}
    for gene in gene_all:
        gene_type=''
        if(gene in new_regenerative_genes):
            gene_type='new_regenerative_gene'
        elif(gene in gene_id_positive):
            gene_type='regenerative_gene'
        elif(gene in regulatory_factor_genes):
            gene_type='regulatory_factor_gene'
        else:
            gene_type = 'candidate_regenerative_gene'
        gene_nodes[gene] = {'gene_id':gene ,'disturb_emb':gene_disturb[gene], 'model_emb':gene_tensor[gene],'gene_type':gene_type}
        for feature in gene_features_str:
            # print(gene_features_data[gene])
            gene_nodes[gene][feature] = gene_features_data[gene][gene_features_str.index(feature)]
    print(len(gene_nodes))
    with open('../result/gene_nodes.pkl','wb')as file:
        pickle.dump(gene_nodes, file)
    pass

In [ ]:
getGenes()
makeGeneEdge()
makeGeneNodes()